# Figshare vs MaterialsCloud spectra comparison

This notebook compares a Figshare key extraction with the MaterialsCloud OmniXAS FEFF raw dataset.

It is concise and sampled by default. It saves CSV and PNG summaries under:

`tutorial_omnixas/analysis_outputs/figshare_vs_materialscloud/`

Set these environment variables if your paths differ:

- `OMNIXAS_DATA_ROOT`
- `ANIONXAS_RAW_ROOT`
- `OMNIXAS_MATERIALSCLOUD_FEFF_ROOT`

Use the native 200-point extraction for Figshare. Do not run a separate 141-point extraction for this notebook.


In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path
import re
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 180,
    "axes.spines.top": False,
    "axes.spines.right": False,
})


In [ ]:
# Configuration.

CWD = Path.cwd()
REPO_ROOT = CWD if (CWD / "tutorial_omnixas").is_dir() else CWD.parent
TUTORIAL_DIR = REPO_ROOT / "tutorial_omnixas"
DATA_ROOT = Path(os.environ.get("OMNIXAS_DATA_ROOT", Path.home() / "OmniXAS_data"))

FIGSHARE_ROOT = Path(os.environ.get("ANIONXAS_RAW_ROOT", DATA_ROOT / "anionxas_curated_200"))
MATERIALSCLOUD_FEFF_ROOT = Path(
    os.environ.get(
        "OMNIXAS_MATERIALSCLOUD_FEFF_ROOT",
        DATA_ROOT / "materialscloud_omnixas_raw" / "extracted" / "FEFF",
    )
)

# Fallback for small local examples.
LOCAL_TUTORIAL_FEFF = TUTORIAL_DIR / "FEFF"
if not MATERIALSCLOUD_FEFF_ROOT.exists() and LOCAL_TUTORIAL_FEFF.exists():
    MATERIALSCLOUD_FEFF_ROOT = LOCAL_TUTORIAL_FEFF

OUTPUT_DIR = TUTORIAL_DIR / "analysis_outputs" / "figshare_vs_materialscloud"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SAMPLE_PER_ELEMENT = 300
RANDOM_SEED = 42
RUN_FULL_SCAN_FOR_MEANS = False
PLOT_GRID = np.linspace(0.0, 35.0, 141)  # common grid used only for comparison plots

OMNIXAS_E_START = {
    "Co": 7709.282,
    "Cr": 5989.168,
    "Cu": 8983.173,
    "Fe": 7111.230,
    "Mn": 6537.886,
    "Ni": 8332.181,
    "Ti": 4964.504,
    "V": 5464.097,
}
BOHR_RADIUS_ANGSTROM = 0.529177210903

print("Figshare root:", FIGSHARE_ROOT)
print("MaterialsCloud FEFF root:", MATERIALSCLOUD_FEFF_ROOT)
print("Output dir:", OUTPUT_DIR)


In [ ]:
def resolve_figshare_feff_root(root: Path) -> Path:
    for candidate in (root / "extracted" / "FEFF", root / "FEFF", root):
        if candidate.is_dir() and any(candidate.glob("*/")):
            return candidate
    return root / "extracted" / "FEFF"


def parse_site_folder(name: str):
    match = re.fullmatch(r"(\d+)_([A-Z][a-z]?)", name)
    if not match:
        return None
    return int(match.group(1)), match.group(2)


def collect_figshare_sites(root: Path) -> pd.DataFrame:
    feff_root = resolve_figshare_feff_root(root)
    rows = []
    if not feff_root.exists():
        warnings.warn(f"Figshare FEFF root not found: {feff_root}")
        return pd.DataFrame(columns=["dataset", "element", "material_id", "site", "path", "has_target_200"])
    for raw_path in feff_root.glob("*/*/FEFF-XANES/*/spectrum_raw.dat"):
        site_info = parse_site_folder(raw_path.parent.name)
        if site_info is None:
            continue
        site, site_element = site_info
        rows.append({
            "dataset": "Figshare",
            "element": raw_path.parents[3].name,
            "material_id": raw_path.parents[2].name,
            "site": site,
            "site_element": site_element,
            "path": raw_path,
            "has_target_200": (raw_path.parent / "spectrum_200.dat").is_file(),
        })
    return pd.DataFrame(rows)


def collect_materialscloud_sites(root: Path) -> pd.DataFrame:
    rows = []
    if not root.exists():
        warnings.warn(f"MaterialsCloud FEFF root not found: {root}")
        return pd.DataFrame(columns=["dataset", "element", "material_id", "site", "path", "has_target_200"])
    for xmu_path in root.glob("*/*/FEFF-XANES/*/xmu.dat"):
        site_info = parse_site_folder(xmu_path.parent.name)
        if site_info is None:
            continue
        site, site_element = site_info
        rows.append({
            "dataset": "MaterialsCloud",
            "element": xmu_path.parents[3].name,
            "material_id": xmu_path.parents[2].name,
            "site": site,
            "site_element": site_element,
            "path": xmu_path,
            "has_target_200": False,
        })
    return pd.DataFrame(rows)


figshare_sites = collect_figshare_sites(FIGSHARE_ROOT)
materialscloud_sites = collect_materialscloud_sites(MATERIALSCLOUD_FEFF_ROOT)
all_sites = pd.concat([figshare_sites, materialscloud_sites], ignore_index=True)

print("Figshare sites:", len(figshare_sites))
print("MaterialsCloud sites:", len(materialscloud_sites))
print("Figshare elements:", sorted(figshare_sites.element.unique())[:20], "...")
print("MaterialsCloud elements:", sorted(materialscloud_sites.element.unique()))


In [ ]:
# Count summary and count plots.

count_summary = (
    all_sites.groupby(["dataset", "element"], observed=True)
    .agg(site_count=("path", "size"), material_count=("material_id", "nunique"))
    .reset_index()
    .sort_values(["dataset", "element"])
)
count_summary.to_csv(OUTPUT_DIR / "count_summary_by_element.csv", index=False)
display(count_summary.head(30))

if not count_summary.empty:
    pivot = count_summary.pivot(index="element", columns="dataset", values="site_count").fillna(0)
    pivot = pivot.sort_index()
    ax = pivot.plot(kind="bar", figsize=(max(10, 0.22 * len(pivot)), 4.5), logy=True)
    ax.set_ylabel("site spectra count, log scale")
    ax.set_title("Available site spectra by element")
    ax.legend(title="dataset")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "site_counts_by_element.png")
    plt.show()

    material_pivot = count_summary.pivot(
        index="element", columns="dataset", values="material_count"
    ).fillna(0).sort_index()
    ax = material_pivot.plot(
        kind="bar", figsize=(max(10, 0.22 * len(material_pivot)), 4.5), logy=True
    )
    ax.set_ylabel("unique material count, log scale")
    ax.set_title("Available unique materials by element")
    ax.legend(title="dataset")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "material_counts_by_element.png")
    plt.show()


In [ ]:
def sample_sites(df: pd.DataFrame, n_per_element: int, seed: int) -> pd.DataFrame:
    if df.empty or RUN_FULL_SCAN_FOR_MEANS:
        return df.copy()
    return (
        df.groupby("element", group_keys=False, observed=True)
        .apply(lambda group: group.sample(min(len(group), n_per_element), random_state=seed))
        .reset_index(drop=True)
    )


def load_figshare_curve(path: Path):
    target_path = path.parent / "spectrum_200.dat"
    if not target_path.is_file():
        raise FileNotFoundError(
            f"Missing spectrum_200.dat for {path.parent}. "
            "Run extraction with anionxas_targets_200.npz. Raw spectra are provenance only."
        )
    data = np.loadtxt(target_path)
    y = np.interp(PLOT_GRID, data[:, 0], data[:, 1])
    return PLOT_GRID, y, {"source": "spectrum_200.dat", "native_points": len(data)}


def read_xsedge_normalization(path: Path) -> float:
    with path.open(encoding="utf-8", errors="replace") as handle:
        for line in handle:
            if line.startswith("#  xsedge+") or line.startswith("# xsedge+"):
                return float(line.split()[-1])
    return 1.0


def load_materialscloud_curve(path: Path, element: str):
    if element not in OMNIXAS_E_START:
        raise ValueError(f"No OmniXAS e_start configured for {element}")
    normalization = read_xsedge_normalization(path)
    data = np.loadtxt(path)
    if data.ndim != 2 or data.shape[1] < 4:
        raise ValueError(f"Bad MaterialsCloud xmu.dat shape {data.shape}: {path}")
    energy = data[:, 0]
    mu = data[:, 3] * normalization / (BOHR_RADIUS_ANGSTROM ** 2)
    good = mu > 0
    energy = energy[good]
    mu = mu[good]
    absolute_grid = OMNIXAS_E_START[element] + PLOT_GRID
    interp = np.interp(absolute_grid, energy, mu)
    return PLOT_GRID, interp, {"source": "xmu.dat", "native_points": len(data), "normalization": normalization}


def normalized_curve(values: np.ndarray) -> np.ndarray:
    values = np.asarray(values, dtype=float)
    high = np.nanmax(values)
    if not np.isfinite(high) or high <= 0:
        return np.full_like(values, np.nan, dtype=float)
    return values / high


In [ ]:
# Load sampled spectra onto a common 141-point relative grid.

figshare_sample = sample_sites(figshare_sites, SAMPLE_PER_ELEMENT, RANDOM_SEED)
materialscloud_sample = sample_sites(materialscloud_sites, SAMPLE_PER_ELEMENT, RANDOM_SEED)

curve_rows = []
curve_arrays = []
failures = []

for _, row in figshare_sample.iterrows():
    try:
        _, y, meta = load_figshare_curve(Path(row["path"]))
        curve_rows.append({**row.drop(labels=["path"]).to_dict(), **meta, "row": len(curve_arrays)})
        curve_arrays.append(y.astype(float))
    except Exception as exc:
        failures.append({
            "dataset": row["dataset"],
            "element": row["element"],
            "material_id": row["material_id"],
            "site": row["site"],
            "path": str(row["path"]),
            "error": str(exc),
        })

for _, row in materialscloud_sample.iterrows():
    try:
        _, y, meta = load_materialscloud_curve(Path(row["path"]), row["element"])
        curve_rows.append({**row.drop(labels=["path"]).to_dict(), **meta, "row": len(curve_arrays)})
        curve_arrays.append(y.astype(float))
    except Exception as exc:
        failures.append({
            "dataset": row["dataset"],
            "element": row["element"],
            "material_id": row["material_id"],
            "site": row["site"],
            "path": str(row["path"]),
            "error": str(exc),
        })

# Keep the manifest schema stable even when every curve from a dataset fails.
curve_columns = [
    "dataset", "element", "material_id", "site", "site_element",
    "has_target_200", "source", "native_points", "normalization", "row",
]
curves = pd.DataFrame(curve_rows, columns=curve_columns)
Y = np.vstack(curve_arrays) if curve_arrays else np.empty((0, len(PLOT_GRID)))
failure_columns = ["dataset", "element", "material_id", "site", "path", "error"]
failures_df = pd.DataFrame(failures, columns=failure_columns)
curves.to_csv(OUTPUT_DIR / "sampled_spectrum_manifest.csv", index=False)
failures_df.to_csv(OUTPUT_DIR / "spectrum_load_failures.csv", index=False)

print("Loaded sampled curves:", Y.shape)
print("Failures:", len(failures_df))
display(failures_df.head())


In [ ]:
# Sampled spectrum statistics.

if len(curves):
    stats = curves.copy()
    stats["y_min"] = np.nanmin(Y, axis=1)
    stats["y_max"] = np.nanmax(Y, axis=1)
    stats["y_area"] = np.trapz(Y, PLOT_GRID, axis=1)
    spectrum_stats = (
        stats.groupby(["dataset", "element"], observed=True)
        .agg(
            sampled_count=("row", "size"),
            y_min_median=("y_min", "median"),
            y_max_median=("y_max", "median"),
            y_area_median=("y_area", "median"),
            native_points_median=("native_points", "median"),
        )
        .reset_index()
    )
    spectrum_stats.to_csv(OUTPUT_DIR / "sampled_spectrum_stats.csv", index=False)
    display(spectrum_stats.head(30))
else:
    spectrum_stats = pd.DataFrame()
    print("No curves loaded.")


In [ ]:
# Mean max-normalized spectra for shared OmniXAS elements.

shared_elements = sorted(set(figshare_sites.element.unique()) & set(materialscloud_sites.element.unique()) & set(OMNIXAS_E_START))
print("Shared elements:", shared_elements)

if len(curves) and shared_elements:
    fig, axes = plt.subplots(2, 4, figsize=(14, 6), sharex=True, sharey=True)
    axes = axes.ravel()
    comparison_rows = []
    for ax, element in zip(axes, shared_elements):
        element_means = {}
        for dataset, color in [("Figshare", "tab:blue"), ("MaterialsCloud", "tab:orange")]:
            row_ids = curves.index[(curves.dataset == dataset) & (curves.element == element)].to_numpy()
            if len(row_ids) == 0:
                continue
            normalized = np.vstack([normalized_curve(Y[i]) for i in row_ids])
            mean = np.nanmean(normalized, axis=0)
            std = np.nanstd(normalized, axis=0)
            element_means[dataset] = mean
            ax.plot(PLOT_GRID, mean, label=f"{dataset} n={len(row_ids)}", color=color)
            ax.fill_between(PLOT_GRID, mean - std, mean + std, color=color, alpha=0.15, linewidth=0)
        if set(element_means) == {"Figshare", "MaterialsCloud"}:
            mad = float(np.nanmean(np.abs(element_means["Figshare"] - element_means["MaterialsCloud"])))
            corr = float(np.corrcoef(element_means["Figshare"], element_means["MaterialsCloud"])[0, 1])
            comparison_rows.append({"element": element, "mean_abs_diff": mad, "mean_correlation": corr})
            ax.text(0.02, 0.05, f"MAD={mad:.3f}\nr={corr:.3f}", transform=ax.transAxes, fontsize=8)
        ax.set_title(element)
        ax.set_xlim(0, 35)
        ax.set_ylim(bottom=0)
    for ax in axes[len(shared_elements):]:
        ax.axis("off")
    axes[0].legend(fontsize=8)
    fig.supxlabel("relative energy (eV)")
    fig.supylabel("intensity / per-curve max")
    fig.suptitle("Mean sampled spectra, max-normalized per curve")
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "mean_spectra_shared_elements.png")
    plt.show()

    comparison = pd.DataFrame(comparison_rows)
    comparison.to_csv(OUTPUT_DIR / "mean_curve_comparison_shared_elements.csv", index=False)
    display(comparison)
else:
    print("No shared curves available for mean comparison.")


In [ ]:
# Global mean and material overlap.

if len(curves):
    fig, ax = plt.subplots(figsize=(7, 4))
    for dataset, color in [("Figshare", "tab:blue"), ("MaterialsCloud", "tab:orange")]:
        row_ids = curves.index[curves.dataset == dataset].to_numpy()
        if len(row_ids) == 0:
            continue
        normalized = np.vstack([normalized_curve(Y[i]) for i in row_ids])
        mean = np.nanmean(normalized, axis=0)
        std = np.nanstd(normalized, axis=0)
        ax.plot(PLOT_GRID, mean, label=f"{dataset} n={len(row_ids)}", color=color)
        ax.fill_between(PLOT_GRID, mean - std, mean + std, color=color, alpha=0.15, linewidth=0)
    ax.set_xlabel("relative energy (eV)")
    ax.set_ylabel("intensity / per-curve max")
    ax.set_title("Global sampled mean spectrum")
    ax.legend()
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "global_mean_spectra.png")
    plt.show()

overlap_rows = []
for element in sorted(set(figshare_sites.element.unique()) | set(materialscloud_sites.element.unique())):
    fig_mats = set(figshare_sites.loc[figshare_sites.element == element, "material_id"])
    mc_mats = set(materialscloud_sites.loc[materialscloud_sites.element == element, "material_id"])
    overlap_rows.append({
        "element": element,
        "figshare_materials": len(fig_mats),
        "materialscloud_materials": len(mc_mats),
        "overlap_materials": len(fig_mats & mc_mats),
        "figshare_only_materials": len(fig_mats - mc_mats),
        "materialscloud_only_materials": len(mc_mats - fig_mats),
    })
overlap = pd.DataFrame(overlap_rows)
overlap.to_csv(OUTPUT_DIR / "material_overlap_by_element.csv", index=False)
display(overlap.sort_values(["overlap_materials", "element"], ascending=[False, True]).head(30))


In [ ]:
# Final summary.

summary = {
    "figshare_root": str(FIGSHARE_ROOT),
    "materialscloud_feff_root": str(MATERIALSCLOUD_FEFF_ROOT),
    "sample_per_element": SAMPLE_PER_ELEMENT,
    "run_full_scan_for_means": RUN_FULL_SCAN_FOR_MEANS,
    "figshare_site_count": int(len(figshare_sites)),
    "materialscloud_site_count": int(len(materialscloud_sites)),
    "loaded_sampled_curves": int(len(curves)),
    "load_failures": int(len(failures_df)),
    "outputs": sorted(path.name for path in OUTPUT_DIR.glob("*")),
}
(OUTPUT_DIR / "summary.json").write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")
print(json.dumps(summary, indent=2))
